# OCR Backend Comparison Colab Workflow

This notebook trains and evaluates a DTD fine-tuning experiment with one configurable OCR prior:

- `none`: no OCR baseline
- `easyocr`: existing EasyOCR prior
- `tesseract`: new Tesseract prior

The goal is a controlled comparison. All arms must use the same manifest files, seed, initial checkpoint, training settings, and evaluation threshold.

Important: this notebook is a fine-tuning ablation workflow. It does not reproduce the full official DTD training curriculum.

## 1. Open This Notebook and Select a GPU

1. Open Google Colab.
2. Upload this notebook or open it from the repository on GitHub.
3. Click `Runtime > Change runtime type`.
4. Select `GPU`.
5. Run the next cell to verify CUDA.

In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Select Runtime > Change runtime type > GPU, then restart runtime.')

## 2. Configuration

Edit this cell first.

Recommended modes:

- `fcd_debug`: fastest prototype. Downloads/extracts only FCD. Good for checking code, not final reporting.
- `temp_train_test`: better scientific protocol without Drive storage. Downloads the Kaggle zip to Colab temporary disk, extracts only `TrainingSet` and `TestingSet`, deletes the zip, and uses train/val from TrainingSet plus test from TestingSet.
- `drive_full`: same protocol as `temp_train_test`, but expects the LMDB folders to already exist in Google Drive.

For the next proper reduced experiment, use `DATA_MODE='temp_train_test'`. Expected setup runtime is roughly 10-25 minutes, mostly download/extract time.

In [ ]:
CONFIG = {
    # Repositories and Drive folders
    'PROJECT_REPO_URL': 'https://github.com/SamiraAbedini/HLCV-Project.git',
    'PROJECT_BRANCH': 'colab-tesseract-workflow',
    'PROJECT_DIR': '/content/HLCV-Project',
    'DOCTAMPER_DIR': '/content/DocTamper',
    'DRIVE_ROOT': '/content/drive/MyDrive/HLCV',

    # Data mode:
    #   fcd_debug       = small prototype using DocTamperV1-FCD downloaded to /content.
    #   temp_train_test = proper reduced train/val/test using TrainingSet + TestingSet on /content.
    #   drive_full      = proper reduced train/val/test using TrainingSet + TestingSet already in Drive.
    'DATA_MODE': 'fcd_debug',
    'KAGGLE_DATASET': 'dinmkeljiame/doctamper',
    'CLEAN_FCD_DEBUG_BEFORE_TEMP_FULL': True,

    # These are set automatically by the setup cell based on DATA_MODE, but may be overridden.
    'DATA_ROOT': '/content/doctamper_data',
    'CHECKPOINT_DIR': '/content/drive/MyDrive/HLCV/checkpoints',
    'MANIFEST_DIR': '/content/drive/MyDrive/HLCV/manifests/fcd_debug_seed42',
    'OUTPUT_ROOT': '/content/drive/MyDrive/HLCV/runs/fcd_debug_tesseract',

    # Reduced dataset manifests
    'SEED': 42,
    'TRAIN_SOURCE': 'DocTamperV1-FCD',
    'TEST_SOURCE': 'DocTamperV1-FCD',
    'TRAIN_SIZE': 80,
    'VAL_SIZE': 20,
    'TEST_SIZE': 20,
    'REGENERATE_MANIFESTS': True,

    # Training settings shared by every arm
    'OCR_BACKEND': 'tesseract',   # choose: none, easyocr, tesseract
    'RUN_ALL_BACKENDS': False,    # set True to run none/easyocr/tesseract sequentially
    'BATCH_SIZE': 2,
    'TRAIN_STEPS': 20,
    'LEARNING_RATE': 1e-4,
    'WEIGHT_DECAY': 1e-2,
    'EVAL_THRESHOLD': 0.5,
    'JPEG_QUALITY': 75,
    'NUM_WORKERS': 0,
    'SAVE_EVERY_STEPS': 10,
    'RESUME': True,

    # OCR settings
    'OCR_CONFIDENCE_THRESHOLD': 30.0,  # Tesseract uses 0-100; EasyOCR uses 0-1.
    'OCR_DILATION': 2,
    'OCR_LANGUAGES': ('eng',),
    'TESSERACT_CMD': None,
    'TESSERACT_PSM': 6,
    'TESSERACT_OEM': 3,

    # Initial checkpoint used by every arm
    'INIT_CHECKPOINT': 'dtd_doctamper.pth',
}

# Proper reduced experiment without storing full LMDBs in Google Drive.
# Uncomment this block when you are ready to run TrainingSet/TestingSet instead of FCD debug.
# CONFIG.update({
#     'DATA_MODE': 'temp_train_test',
#     'DATA_ROOT': '/content/doctamper_train_test',
#     'MANIFEST_DIR': '/content/drive/MyDrive/HLCV/manifests/train800_val200_test200_seed42',
#     'OUTPUT_ROOT': '/content/drive/MyDrive/HLCV/runs/train800_val200_test200_tesseract',
#     'TRAIN_SOURCE': 'DocTamperV1-TrainingSet',
#     'TEST_SOURCE': 'DocTamperV1-TestingSet',
#     'TRAIN_SIZE': 800,
#     'VAL_SIZE': 200,
#     'TEST_SIZE': 200,
#     'BATCH_SIZE': 2,
#     'TRAIN_STEPS': 400,
#     'NUM_WORKERS': 0,
#     'SAVE_EVERY_STEPS': 100,
# })

# Full Drive-backed version, if you later have enough Drive space.
# CONFIG.update({
#     'DATA_MODE': 'drive_full',
#     'DATA_ROOT': '/content/drive/MyDrive/HLCV/doctamper_data',
#     'MANIFEST_DIR': '/content/drive/MyDrive/HLCV/manifests/colab_2400_seed42',
#     'OUTPUT_ROOT': '/content/drive/MyDrive/HLCV/runs/ocr_backend_comparison',
#     'TRAIN_SOURCE': 'DocTamperV1-TrainingSet',
#     'TEST_SOURCE': 'DocTamperV1-TestingSet',
#     'TRAIN_SIZE': 1600,
#     'VAL_SIZE': 400,
#     'TEST_SIZE': 400,
#     'BATCH_SIZE': 4,
#     'TRAIN_STEPS': 800,
#     'NUM_WORKERS': 2,
#     'SAVE_EVERY_STEPS': 100,
# })

CONFIG

## 3. Mount Google Drive

Colab sessions disappear when they disconnect. Google Drive stores the dataset, manifests, checkpoints, cache, and results persistently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
for key in ['DRIVE_ROOT', 'OUTPUT_ROOT', 'MANIFEST_DIR']:
    Path(CONFIG[key]).mkdir(parents=True, exist_ok=True)
print('Drive folders ready.')

## 4. Install Dependencies Including Tesseract

This installs the Python packages, the Tesseract system executable, the English language pack, and the `jpegio` version expected by DTD.

In [ ]:
%cd /content
!pip -q install lmdb six pillow opencv-python-headless tqdm matplotlib scikit-learn kaggle albumentations timm==0.4.12 segmentation_models_pytorch==0.2.1 easyocr pytesseract
!pip -q install efficientnet_pytorch==0.7.1
!apt-get update -qq > /dev/null
!apt-get install -y -qq tesseract-ocr tesseract-ocr-eng libjpeg-dev > /dev/null
!tesseract --version | head -n 2
!tesseract --list-langs

!pip -q uninstall -y jpegio
!rm -rf /content/jpegio
!git clone -q https://github.com/dwgoon/jpegio.git /content/jpegio
%cd /content/jpegio
!pip -q install .
%cd /content

## 5. Clone Repositories and Make Imports Work

This cell always checks out the branch named in `CONFIG['PROJECT_BRANCH']`. That matters because the notebook imports the new `src/` and `scripts/` files from this branch.

In [ ]:
import os, sys, subprocess
from pathlib import Path

%cd /content

project = Path(CONFIG['PROJECT_DIR'])
branch = CONFIG.get('PROJECT_BRANCH', 'main')

if project.exists():
    subprocess.run(['git', '-C', str(project), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(project), 'checkout', branch], check=True)
    subprocess.run(['git', '-C', str(project), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', branch, CONFIG['PROJECT_REPO_URL'], str(project)], check=True)

doctamper = Path(CONFIG['DOCTAMPER_DIR'])
if not doctamper.exists():
    subprocess.run(['git', 'clone', 'https://github.com/qcf-568/DocTamper.git', str(doctamper)], check=True)

for p in [str(project), str(doctamper / 'models')]:
    if p not in sys.path:
        sys.path.insert(0, p)

active_branch = subprocess.check_output(['git', '-C', str(project), 'branch', '--show-current'], text=True).strip()
print('Project path:', project)
print('Project branch:', active_branch)
print('DocTamper path:', doctamper)

## 6. Prepare Data, Manifests, and Checkpoints

Expected runtime:

- `fcd_debug`: 1-8 minutes if FCD is already extracted, longer if it must download.
- `temp_train_test`: 10-25 minutes because it downloads the Kaggle zip and extracts only TrainingSet + TestingSet.
- `drive_full`: under 2 minutes if data is already in Drive.

This cell:

1. Checks and stages the three DTD checkpoint files from `MyDrive/HLCV/checkpoints`.
2. Prepares the dataset according to `DATA_MODE`.
3. Deletes the huge Kaggle zip after extraction.
4. Generates and validates disjoint train/val/test manifests.

`temp_train_test` is the recommended path when Drive cannot store the full dataset.

In [ ]:
from pathlib import Path
import getpass
import os
import shutil
import subprocess

DATA_MODE = CONFIG.get('DATA_MODE', 'fcd_debug')
DRIVE_ROOT = Path(CONFIG['DRIVE_ROOT'])
CKPT_DIR = Path(CONFIG['CHECKPOINT_DIR'])
MODELS_DIR = Path(CONFIG['DOCTAMPER_DIR']) / 'models'
PROJECT_DIR = Path(CONFIG['PROJECT_DIR'])

if DATA_MODE == 'fcd_debug':
    CONFIG.update({
        'DATA_ROOT': CONFIG.get('DATA_ROOT', '/content/doctamper_data'),
        'TRAIN_SOURCE': 'DocTamperV1-FCD',
        'TEST_SOURCE': 'DocTamperV1-FCD',
        'MANIFEST_DIR': str(DRIVE_ROOT / 'manifests' / 'fcd_debug_seed42'),
        'OUTPUT_ROOT': str(DRIVE_ROOT / 'runs' / 'fcd_debug_tesseract'),
    })
elif DATA_MODE == 'temp_train_test':
    CONFIG.update({
        'DATA_ROOT': CONFIG.get('DATA_ROOT', '/content/doctamper_train_test'),
        'TRAIN_SOURCE': 'DocTamperV1-TrainingSet',
        'TEST_SOURCE': 'DocTamperV1-TestingSet',
    })
    CONFIG.setdefault('MANIFEST_DIR', str(DRIVE_ROOT / 'manifests' / 'train800_val200_test200_seed42'))
    CONFIG.setdefault('OUTPUT_ROOT', str(DRIVE_ROOT / 'runs' / 'train800_val200_test200_tesseract'))
elif DATA_MODE == 'drive_full':
    CONFIG.setdefault('DATA_ROOT', str(DRIVE_ROOT / 'doctamper_data'))
else:
    raise ValueError(f"Unknown DATA_MODE: {DATA_MODE}")

DATA_ROOT = Path(CONFIG['DATA_ROOT'])
MANIFEST_DIR = Path(CONFIG['MANIFEST_DIR'])

print('DATA_MODE:', DATA_MODE)
print('DATA_ROOT:', DATA_ROOT)
print('CHECKPOINT_DIR:', CKPT_DIR)
print('MANIFEST_DIR:', MANIFEST_DIR)
print('OUTPUT_ROOT:', CONFIG['OUTPUT_ROOT'])
print('TRAIN/VAL/TEST sizes:', CONFIG['TRAIN_SIZE'], CONFIG['VAL_SIZE'], CONFIG['TEST_SIZE'])

# Checkpoints are not committed to Git. Put them in Drive once.
for fname in ['vph_imagenet.pt', 'swin_imagenet.pt', CONFIG['INIT_CHECKPOINT']]:
    src = CKPT_DIR / fname
    assert src.exists(), f"Missing checkpoint: {src}. Download it into MyDrive/HLCV/checkpoints first."
    dst = MODELS_DIR / fname
    if (not dst.exists()) or (dst.stat().st_size != src.stat().st_size):
        if dst.exists():
            print('Replacing staged checkpoint with Drive copy:', fname)
            dst.unlink()
        shutil.copy2(src, dst)
        print('Copied checkpoint:', fname)
    else:
        print('Checkpoint already staged:', fname)

qt_src = Path(CONFIG['DOCTAMPER_DIR']) / 'qt_table.pk'
qt_dst = MODELS_DIR / 'qt_table.pk'
if not qt_dst.exists():
    shutil.copy2(qt_src, qt_dst)
    print('Copied qt_table.pk')
else:
    print('qt_table.pk already staged')

def ensure_kaggle_token():
    token_file = Path.home() / '.kaggle' / 'access_token'
    if not token_file.exists() and not os.environ.get('KAGGLE_API_TOKEN'):
        token = getpass.getpass('Paste Kaggle API token (hidden): ')
        token_file.parent.mkdir(parents=True, exist_ok=True)
        token_file.write_text(token.strip())
        token_file.chmod(0o600)
        print('Saved Kaggle token to', token_file)

def download_kaggle_zip():
    ensure_kaggle_token()
    tmp_download = Path('/content/doctamper_kaggle')
    tmp_download.mkdir(parents=True, exist_ok=True)
    zip_path = tmp_download / 'doctamper.zip'
    if not zip_path.exists():
        print('Downloading Kaggle DocTamper zip to temporary Colab disk...')
        subprocess.run(['kaggle', 'datasets', 'download', '-d', CONFIG['KAGGLE_DATASET'], '-p', str(tmp_download)], check=True)
    print('Zip ready:', zip_path, zip_path.stat().st_size)
    return zip_path

def extract_patterns(zip_path, patterns):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    for pattern in patterns:
        print('Extracting pattern:', pattern)
        subprocess.run(['unzip', '-o', str(zip_path), pattern, '-d', str(DATA_ROOT)], check=True)

# Prepare dataset.
if DATA_MODE == 'fcd_debug':
    fcd_dir = DATA_ROOT / 'DocTamperV1-FCD'
    if not (fcd_dir / 'data.mdb').exists():
        zip_path = download_kaggle_zip()
        extract_patterns(zip_path, ['DocTamperV1-FCD/*'])
        if zip_path.exists():
            print('Deleting downloaded zip to free disk:', zip_path)
            zip_path.unlink()
    assert (fcd_dir / 'data.mdb').exists(), f'FCD extraction failed: {fcd_dir}'
elif DATA_MODE == 'temp_train_test':
    if CONFIG.get('CLEAN_FCD_DEBUG_BEFORE_TEMP_FULL', True):
        old_fcd = Path('/content/doctamper_data/DocTamperV1-FCD')
        if old_fcd.exists():
            print('Removing old FCD debug data to free space:', old_fcd)
            shutil.rmtree(old_fcd)
    train_dir = DATA_ROOT / 'DocTamperV1-TrainingSet'
    test_dir = DATA_ROOT / 'DocTamperV1-TestingSet'
    if not ((train_dir / 'data.mdb').exists() and (test_dir / 'data.mdb').exists()):
        free_gb = shutil.disk_usage('/content').free / (1024 ** 3)
        print(f'Free /content disk before download: {free_gb:.1f} GB')
        zip_path = download_kaggle_zip()
        extract_patterns(zip_path, ['DocTamperV1-TrainingSet/*', 'DocTamperV1-TestingSet/*'])
        if zip_path.exists():
            print('Deleting downloaded zip to free disk:', zip_path)
            zip_path.unlink()
    assert (train_dir / 'data.mdb').exists(), f'TrainingSet extraction failed: {train_dir}'
    assert (test_dir / 'data.mdb').exists(), f'TestingSet extraction failed: {test_dir}'
else:
    for split in [CONFIG['TRAIN_SOURCE'], CONFIG['TEST_SOURCE']]:
        assert (DATA_ROOT / split / 'data.mdb').exists(), f'Missing dataset LMDB: {DATA_ROOT / split}'

print('Disk after dataset prep:')
subprocess.run(['df', '-h', '/content'], check=True)

# Generate deterministic manifests.
manifest_files = [MANIFEST_DIR / f'{s}.json' for s in ['train', 'val', 'test']]
need_manifests = CONFIG.get('REGENERATE_MANIFESTS', False) or not all(p.exists() for p in manifest_files)
if need_manifests:
    cmd = [
        'python', str(PROJECT_DIR / 'scripts' / 'generate_doctamper_subset.py'),
        '--data-root', str(DATA_ROOT),
        '--output-dir', str(MANIFEST_DIR),
        '--seed', str(CONFIG['SEED']),
        '--train-source', CONFIG['TRAIN_SOURCE'],
        '--test-source', CONFIG['TEST_SOURCE'],
        '--train-size', str(CONFIG['TRAIN_SIZE']),
        '--val-size', str(CONFIG['VAL_SIZE']),
        '--test-size', str(CONFIG['TEST_SIZE']),
    ]
    if DATA_MODE == 'fcd_debug':
        cmd.append('--allow-eval-source-for-training')
    subprocess.run(cmd, check=True)
else:
    print('Using existing manifests:', MANIFEST_DIR)

subprocess.run([
    'python', str(PROJECT_DIR / 'scripts' / 'validate_doctamper_subset.py'),
    '--data-root', str(DATA_ROOT),
    str(MANIFEST_DIR / 'train.json'),
    str(MANIFEST_DIR / 'val.json'),
    str(MANIFEST_DIR / 'test.json'),
], check=True)

if DATA_MODE == 'fcd_debug':
    print('WARNING: FCD debug mode is for code verification only, not final scientific reporting.')
elif DATA_MODE == 'temp_train_test':
    print('Using TrainingSet for train/val and TestingSet for test. This is the recommended reduced protocol.')
print('Data/checkpoint setup complete.')

## 7. Load the Reduced Dataset

This uses `ManifestDocTamperDataset`, which reads only the listed LMDB indices. The original dataset is unchanged.

Assumption: because the public DocTamper repo does not include TrainingSet compression-record pickles, this workflow uses a fixed JPEG quality table (`JPEG_QUALITY`) for reproducible fine-tuning.

In [ ]:
import random, numpy as np, torch
from torch.utils.data import DataLoader
from src.doctamper_dataset import ManifestDocTamperDataset
from src.doctamper_lmdb import load_manifest

random.seed(CONFIG['SEED'])
np.random.seed(CONFIG['SEED'])
torch.manual_seed(CONFIG['SEED'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

train_manifest = load_manifest(MANIFEST_DIR / 'train.json')
val_manifest = load_manifest(MANIFEST_DIR / 'val.json')
test_manifest = load_manifest(MANIFEST_DIR / 'test.json')
qt_path = MODELS_DIR / 'qt_table.pk'

train_ds = ManifestDocTamperDataset(DATA_ROOT, train_manifest, qt_path, jpeg_quality=CONFIG['JPEG_QUALITY'])
val_ds = ManifestDocTamperDataset(DATA_ROOT, val_manifest, qt_path, jpeg_quality=CONFIG['JPEG_QUALITY'])
test_ds = ManifestDocTamperDataset(DATA_ROOT, test_manifest, qt_path, jpeg_quality=CONFIG['JPEG_QUALITY'])

train_loader = DataLoader(train_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=True, num_workers=CONFIG['NUM_WORKERS'], drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=CONFIG['NUM_WORKERS'])
test_loader = DataLoader(test_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=CONFIG['NUM_WORKERS'])

print('train/val/test:', len(train_ds), len(val_ds), len(test_ds))
batch = next(iter(train_loader))
print('image', batch['image'].shape, 'label', batch['label'].shape, 'dct', batch['rgb'].shape, 'q', batch['q'].shape)

## 8. Preview Image and Mask Pairs

This catches wrong paths or broken masks before training starts.

In [ ]:
import matplotlib.pyplot as plt
MEAN = np.array([0.485, 0.455, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def denorm_tensor(img):
    arr = img.permute(1, 2, 0).cpu().numpy() * STD + MEAN
    return np.clip(arr * 255, 0, 255).astype(np.uint8)

batch = next(iter(train_loader))
n = min(4, batch['image'].shape[0])
plt.figure(figsize=(8, 4*n))
for i in range(n):
    plt.subplot(n, 2, 2*i+1); plt.imshow(denorm_tensor(batch['image'][i])); plt.title(batch['sample_id'][i]); plt.axis('off')
    plt.subplot(n, 2, 2*i+2); plt.imshow(batch['label'][i,0].numpy(), cmap='gray'); plt.title('GT tamper mask'); plt.axis('off')
plt.tight_layout(); plt.show()

## 9. OCR Backend and Cache

OCR is slow, especially Tesseract on CPU. This cache stores detections using a key derived from backend, language, confidence threshold, PSM/OEM, and dilation, so EasyOCR and Tesseract cannot accidentally share masks.

In [ ]:
from src.ocr_backends import OCRConfig, create_ocr_backend
from src.ocr_cache import OCRDetectionCache

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def make_ocr(backend_name):
    if backend_name == 'easyocr':
        langs = tuple('en' if x == 'eng' else x for x in CONFIG['OCR_LANGUAGES'])
        conf = 0.0 if CONFIG['OCR_CONFIDENCE_THRESHOLD'] > 1 else CONFIG['OCR_CONFIDENCE_THRESHOLD']
    else:
        langs = CONFIG['OCR_LANGUAGES']
        conf = CONFIG['OCR_CONFIDENCE_THRESHOLD']
    cfg = OCRConfig(
        backend=backend_name,
        languages=tuple(langs),
        confidence_threshold=conf,
        dilation=CONFIG['OCR_DILATION'],
        easyocr_gpu=torch.cuda.is_available(),
        tesseract_cmd=CONFIG['TESSERACT_CMD'],
        tesseract_psm=CONFIG['TESSERACT_PSM'],
        tesseract_oem=CONFIG['TESSERACT_OEM'],
    )
    backend = create_ocr_backend(cfg)
    cache = OCRDetectionCache(Path(CONFIG['OUTPUT_ROOT']) / 'ocr_cache', cfg)
    return cfg, backend, cache

current_ocr_cfg, current_ocr_backend, current_ocr_cache = make_ocr(CONFIG['OCR_BACKEND'])
print(current_ocr_cfg)
print('cache dir:', current_ocr_cache.root)

In [ ]:
# Visualize OCR mask on a few validation images.
from src.ocr_eval import OCRCoverageMeter

batch = next(iter(val_loader))
plt.figure(figsize=(12, 3 * min(3, len(batch['image']))))
for i in range(min(3, len(batch['image']))):
    image = denorm_tensor(batch['image'][i])
    detections, hit = current_ocr_cache.get_or_compute(batch['sample_id'][i], image, current_ocr_backend)
    text_mask = current_ocr_cache.mask_from_detections(detections, image.shape)
    gt = batch['label'][i,0].numpy() > 0
    missed = gt & ~(text_mask > 0)
    for j, (im, title, kwargs) in enumerate([
        (image, 'image', {}),
        (text_mask, f'OCR mask boxes={len(detections)} hit={hit}', {'cmap':'gray'}),
        (gt, 'GT tamper', {'cmap':'gray'}),
        (missed, 'missed tamper pixels', {'cmap':'Reds'}),
    ]):
        plt.subplot(min(3, len(batch['image'])), 4, 4*i+j+1); plt.imshow(im, **kwargs); plt.title(title); plt.axis('off')
plt.tight_layout(); plt.show()

## 10. Build DTD Model and Optional Prior Fusion

All arms start from the same checkpoint. `none` does not wrap the segmentation head. `easyocr` and `tesseract` use the same fusion block and differ only in OCR detections.

In [ ]:
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler

import os
os.chdir(MODELS_DIR)
print('Working directory:', Path.cwd())
_ORIG_LOAD = torch.load
def _safe_load(*a, **k):
    k.setdefault('weights_only', False)
    return _ORIG_LOAD(*a, **k)
torch.load = _safe_load
from dtd import *
from src.losses import CombinedTamperLoss
from src.fusion import TextPriorFusion

criterion = CombinedTamperLoss(lambda_dice=1.0, lambda_bound=0.5)

def build_model():
    model = seg_dtd('', 2).to(DEVICE)
    for mod in model.modules():
        if isinstance(mod, nn.GELU) and not hasattr(mod, 'approximate'):
            mod.approximate = 'none'
    ckpt_path = MODELS_DIR / CONFIG['INIT_CHECKPOINT']
    sd = torch.load(ckpt_path, map_location='cpu')['state_dict']
    sd = {k.replace('module.', ''): v for k, v in sd.items()}
    model.load_state_dict(sd, strict=False)
    return model

def forward_dtd(model, batch):
    return model(batch['image'].to(DEVICE), batch['rgb'].to(DEVICE), batch['q'].unsqueeze(1).to(DEVICE))

class HeadWithPrior(nn.Module):
    def __init__(self, head, in_ch):
        super().__init__()
        self.fusion = TextPriorFusion(in_ch)
        self.head = head
        self.text_mask = None
    def forward(self, feat):
        return self.head(self.fusion(feat, self.text_mask))

def wire_ocr_prior(model):
    core = model.model
    head = core.segmentation_head
    while hasattr(head, 'head'):
        head = head.head
    core.segmentation_head = HeadWithPrior(head, head[0].in_channels).to(DEVICE)
    return core

print('Model helpers ready on', DEVICE)

## 11. Training, Resume, and Evaluation Helpers

If Colab disconnects, rerun setup cells and this cell, then keep `RESUME=True`. The latest checkpoint in the experiment folder will be loaded.

In [ ]:
from sklearn.metrics import roc_auc_score
import json, shutil, time

rng_auc = np.random.default_rng(CONFIG['SEED'])

def batch_text_masks(batch, backend, cache):
    masks = []
    for img_tensor, sid in zip(batch['image'], batch['sample_id']):
        image = denorm_tensor(img_tensor)
        detections, _ = cache.get_or_compute(sid, image, backend)
        masks.append(cache.mask_from_detections(detections, image.shape))
    return torch.from_numpy(np.stack(masks)[:, None]).float().to(DEVICE)

def save_json(path, payload):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(payload, f, indent=2, sort_keys=True, default=str)
        f.write('\n')

def train_one(model, loader, backend_name, backend, cache, out_dir):
    core = model.model
    use_prior = backend_name != 'none'
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['LEARNING_RATE'], weight_decay=CONFIG['WEIGHT_DECAY'])
    scaler = GradScaler()
    start_step = 0
    history = []
    latest = Path(out_dir) / 'checkpoints' / 'latest.pth'
    if CONFIG['RESUME'] and latest.exists():
        ckpt = torch.load(latest, map_location=DEVICE)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        start_step = int(ckpt['step']) + 1
        history = ckpt.get('history', [])
        print('Resumed from step', start_step)
    model.train()
    iterator = iter(loader)
    t0 = time.time()
    for step in range(start_step, CONFIG['TRAIN_STEPS']):
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            batch = next(iterator)
        if use_prior:
            core.segmentation_head.text_mask = batch_text_masks(batch, backend, cache)
        target = batch['label'].squeeze(1).long().to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(DEVICE == 'cuda')):
            logits = forward_dtd(model, batch)
            if logits.shape[-2:] != target.shape[-2:]:
                logits = F.interpolate(logits, size=target.shape[-2:], mode='bilinear', align_corners=False)
            loss, parts = criterion(logits, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        rec = {'step': step, 'loss': float(parts['total']), 'ce': float(parts['ce']), 'dice': float(parts['dice']), 'boundary': float(parts['boundary'])}
        history.append(rec)
        if step % 25 == 0:
            print(f"step {step:04d}/{CONFIG['TRAIN_STEPS']} loss={rec['loss']:.4f} ce={rec['ce']:.4f} dice={rec['dice']:.4f} boundary={rec['boundary']:.4f}")
        if (step + 1) % CONFIG['SAVE_EVERY_STEPS'] == 0 or step + 1 == CONFIG['TRAIN_STEPS']:
            latest.parent.mkdir(parents=True, exist_ok=True)
            torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'step': step, 'history': history}, latest)
            torch.save({'state_dict': model.state_dict()}, Path(out_dir) / 'checkpoints' / f'step_{step+1:05d}.pth')
    return history, time.time() - t0

@torch.no_grad()
def evaluate_model(model, loader, backend_name, backend, cache):
    model.eval()
    core = model.model
    use_prior = backend_name != 'none'
    tp = fp = fn = 0
    auc_scores, auc_labels = [], []
    for batch in loader:
        if use_prior:
            core.segmentation_head.text_mask = batch_text_masks(batch, backend, cache)
        target = batch['label'].squeeze(1).long().to(DEVICE)
        with autocast(enabled=(DEVICE == 'cuda')):
            logits = forward_dtd(model, batch)
            if logits.shape[-2:] != target.shape[-2:]:
                logits = F.interpolate(logits, size=target.shape[-2:], mode='bilinear', align_corners=False)
        prob = torch.softmax(logits.float(), 1)[:, 1]
        pred = prob >= CONFIG['EVAL_THRESHOLD']
        gt = target.bool()
        tp += int((pred & gt).sum()); fp += int((pred & ~gt).sum()); fn += int((~pred & gt).sum())
        p = prob.detach().cpu().numpy().reshape(-1)
        y = gt.cpu().numpy().reshape(-1).astype(np.uint8)
        k = min(20000, len(p))
        idx = rng_auc.choice(len(p), size=k, replace=False)
        auc_scores.append(p[idx]); auc_labels.append(y[idx])
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    iou = tp / (tp + fp + fn + 1e-9)
    y = np.concatenate(auc_labels); s = np.concatenate(auc_scores)
    auc = roc_auc_score(y, s) if y.min() != y.max() else float('nan')
    return {'pixel_f1': f1, 'precision': precision, 'recall': recall, 'foreground_iou': iou, 'auc': auc, 'threshold': CONFIG['EVAL_THRESHOLD']}

@torch.no_grad()
def evaluate_ocr_prior(loader, backend_name, backend, cache, max_batches=None):
    from src.ocr_eval import OCRCoverageMeter
    meter = OCRCoverageMeter(coverage_thresh=0.5)
    runtimes, box_counts = [], []
    import time
    for bi, batch in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        for img_tensor, sid, gt_tensor in zip(batch['image'], batch['sample_id'], batch['label']):
            image = denorm_tensor(img_tensor)
            t0 = time.perf_counter()
            detections, hit = cache.get_or_compute(sid, image, backend)
            elapsed = 0.0 if hit else time.perf_counter() - t0
            text_mask = cache.mask_from_detections(detections, image.shape)
            tamper = gt_tensor[0].numpy() > 0
            if tamper.any():
                meter.update(text_mask, tamper)
            runtimes.append(elapsed); box_counts.append(len(detections))
    result = meter.compute()
    result.update({'backend': backend_name, 'runtime_per_image_sec': float(np.mean(runtimes)), 'mean_num_boxes': float(np.mean(box_counts))})
    return result

print('Training and evaluation helpers ready.')

## 12. Run Experiment

For a fair comparison, run the same cell for `none`, `easyocr`, and `tesseract`, or set `RUN_ALL_BACKENDS=True` in the config. Each arm starts from the same initial checkpoint.

In [ ]:
backends_to_run = ['none', 'easyocr', 'tesseract'] if CONFIG['RUN_ALL_BACKENDS'] else [CONFIG['OCR_BACKEND']]
all_results = {}

for backend_name in backends_to_run:
    print('\n=== Running', backend_name, '===')
    run_dir = Path(CONFIG['OUTPUT_ROOT']) / f"{backend_name}_seed{CONFIG['SEED']}"
    run_dir.mkdir(parents=True, exist_ok=True)
    ocr_cfg, ocr_backend, ocr_cache = make_ocr(backend_name)
    model = build_model()
    if backend_name != 'none':
        wire_ocr_prior(model)
    ocr_metrics = evaluate_ocr_prior(val_loader, backend_name, ocr_backend, ocr_cache, max_batches=None)
    save_json(run_dir / 'ocr_metrics.json', ocr_metrics)
    history, train_time = train_one(model, train_loader, backend_name, ocr_backend, ocr_cache, run_dir)
    val_metrics = evaluate_model(model, val_loader, backend_name, ocr_backend, ocr_cache)
    test_metrics = evaluate_model(model, test_loader, backend_name, ocr_backend, ocr_cache)
    save_json(run_dir / 'metrics.json', test_metrics)
    save_json(run_dir / 'val_metrics.json', val_metrics)
    save_json(run_dir / 'history.json', history)
    run_config = dict(CONFIG)
    run_config.update({'ocr_backend': backend_name, 'ocr_config': ocr_cfg.__dict__, 'train_time_sec': train_time, 'train_size': len(train_ds), 'val_size': len(val_ds), 'test_size': len(test_ds)})
    save_json(run_dir / 'config.json', run_config)
    shutil.copy2(MANIFEST_DIR / 'train.json', run_dir / 'train_manifest.json')
    shutil.copy2(MANIFEST_DIR / 'val.json', run_dir / 'val_manifest.json')
    shutil.copy2(MANIFEST_DIR / 'test.json', run_dir / 'test_manifest.json')
    all_results[backend_name] = {'val': val_metrics, 'test': test_metrics, 'ocr': ocr_metrics}
    print('VAL:', {k: round(v, 4) for k, v in val_metrics.items() if isinstance(v, float)})
    print('TEST:', {k: round(v, 4) for k, v in test_metrics.items() if isinstance(v, float)})
    print('OCR:', {k: round(v, 4) for k, v in ocr_metrics.items() if isinstance(v, float)})

## 13. Qualitative Outputs

This saves a few images your teammate can inspect: document image, OCR mask, GT tamper mask, prediction probability, and missed tampered pixels.

In [ ]:
@torch.no_grad()
def save_qualitative(model, loader, backend_name, backend, cache, out_dir, n_images=6):
    model.eval()
    core = model.model
    use_prior = backend_name != 'none'
    out_dir = Path(out_dir) / 'qualitative'
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = 0
    for batch in loader:
        if use_prior:
            core.segmentation_head.text_mask = batch_text_masks(batch, backend, cache)
        with autocast(enabled=(DEVICE == 'cuda')):
            logits = forward_dtd(model, batch)
            target = batch['label'].squeeze(1).long().to(DEVICE)
            if logits.shape[-2:] != target.shape[-2:]:
                logits = F.interpolate(logits, size=target.shape[-2:], mode='bilinear', align_corners=False)
        prob = torch.softmax(logits.float(), 1)[:, 1].cpu().numpy()
        for i in range(len(batch['image'])):
            image = denorm_tensor(batch['image'][i])
            if use_prior:
                detections, _ = cache.get_or_compute(batch['sample_id'][i], image, backend)
                text_mask = cache.mask_from_detections(detections, image.shape)
            else:
                text_mask = np.zeros(image.shape[:2], dtype=np.uint8)
            gt = batch['label'][i,0].numpy() > 0
            missed = gt & (prob[i] < CONFIG['EVAL_THRESHOLD'])
            fig, axes = plt.subplots(1, 5, figsize=(15, 3))
            for ax, im, title, kwargs in [
                (axes[0], image, 'image', {}),
                (axes[1], text_mask, 'OCR mask', {'cmap':'gray'}),
                (axes[2], gt, 'GT tamper', {'cmap':'gray'}),
                (axes[3], prob[i], 'pred prob', {'cmap':'jet', 'vmin':0, 'vmax':1}),
                (axes[4], missed, 'missed tamper', {'cmap':'Reds'}),
            ]:
                ax.imshow(im, **kwargs); ax.set_title(title); ax.axis('off')
            fig.tight_layout()
            fig.savefig(out_dir / f"{saved:03d}_{backend_name}.png", dpi=160)
            plt.close(fig)
            saved += 1
            if saved >= n_images:
                return

# Optional: reload latest model from one run and save qualitative figures.
# This cell assumes the last loop variables still point to the most recent model/backend.
try:
    save_qualitative(model, test_loader, backend_name, ocr_backend, ocr_cache, run_dir, n_images=6)
    print('Saved qualitative figures to', Path(run_dir) / 'qualitative')
except NameError:
    print('Run an experiment first.')

## 14. Generate Comparison Report

After running multiple arms, this creates CSV, JSON, and Markdown outputs that are easy to send to your teammate.

In [ ]:
experiment_dirs = [str(Path(CONFIG['OUTPUT_ROOT']) / f"{b}_seed{CONFIG['SEED']}") for b in ['none', 'easyocr', 'tesseract'] if (Path(CONFIG['OUTPUT_ROOT']) / f"{b}_seed{CONFIG['SEED']}" / 'metrics.json').exists()]
if experiment_dirs:
    report_dir = Path(CONFIG['OUTPUT_ROOT']) / 'comparison_report'
    !python {CONFIG['PROJECT_DIR']}/scripts/compare_experiment_results.py --experiments {' '.join(experiment_dirs)} --output-dir "{report_dir}"
    print('Report:', report_dir / 'comparison.md')
else:
    print('No completed experiment dirs found yet.')

## 15. Safe Stop and Resume

Safe stop:

1. Wait until a checkpoint message appears or manually run the save cell if you added one.
2. Confirm `latest.pth`, `config.json`, and manifests exist in the run folder.
3. Disconnect runtime.

Resume:

1. Reopen notebook.
2. Select GPU.
3. Rerun setup/config cells.
4. Keep `RESUME=True`.
5. Run the experiment cell again.

The notebook resumes from `checkpoints/latest.pth` in the matching backend/seed run folder.

## 16. Common Errors and Fixes

- CUDA not available: select `Runtime > Change runtime type > GPU`, then restart runtime.
- Out of memory: reduce `BATCH_SIZE` first, then `TRAIN_STEPS`, then manifest sizes. Batch size is usually the fastest fix.
- Drive disconnected: remount Drive and rerun path validation.
- Missing dataset files: check `DATA_ROOT` and confirm each split contains `data.mdb` and `lock.mdb`.
- Missing Tesseract executable: rerun the install cell and verify `!tesseract --version`.
- Missing language packs: install the pack, for example `tesseract-ocr-deu`, and set `OCR_LANGUAGES=('deu',)` or `('eng','deu')`.
- Import errors: rerun dependency installation and make sure `PROJECT_DIR` is on `sys.path`.
- Incompatible package versions: restart runtime after reinstalling DTD dependencies.
- Checkpoint-loading errors: verify `vph_imagenet.pt`, `swin_imagenet.pt`, and `dtd_doctamper.pth` are copied from the same checkpoint release.
- Very broad OCR mask: lower dilation or raise confidence threshold. High forged-region coverage is not useful if `text_area_ratio` is huge.
- Tesseract is slow: keep OCR cache on Drive and avoid changing OCR config unless you intentionally want a new cache.